# Lie Symmetry Analysis of the Nonlinear Cubic Klein–Gordon Equation

This tutorial demonstrates using **`symlie`** to analyze the relativistic scalar field equation with cubic self-interaction:
$$u_{tt} - u_{xx} + u^3 = 0$$

We investigate:
1. Variational formulation and the Euler–Lagrange equation.
2. The 4-dimensional translation, Lorentz, and scaling Lie algebra.
3. Symmetries and relativistic Lorentz invariance.
4. Verification of a singular algebraic traveling profile.

In [ ]:
import sympy as sp

from symlie import (
    euler_lagrange,
    infinitesimals,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, t = sp.symbols("x t")
u = sp.Function("u")(x, t)

# Cubic Klein-Gordon Equation
ckg_eq = u.diff(t, 2) - u.diff(x, 2) + u**3
print("PDE Order:", max_derivative_order(ckg_eq, u, (x, t)))
sp.Eq(ckg_eq, 0)

## 1. Variational Formulation (Euler–Lagrange Operator)

The Lagrangian density for the nonlinear Klein–Gordon field is:
$$\mathcal{L} = \frac{1}{2} u_t^2 - \frac{1}{2} u_x^2 - \frac{1}{4} u^4$$

We verify with `euler_lagrange`:

In [ ]:
L = (
    sp.Rational(1, 2) * u.diff(t) ** 2
    - sp.Rational(1, 2) * u.diff(x) ** 2
    - sp.Rational(1, 4) * u**4
)
el_res = euler_lagrange(L, u, (x, t))

print("Euler-Lagrange Variational Derivative E_u(L):")
display(el_res[0])
assert sp.simplify(el_res[0] + ckg_eq) == 0
print("Confirmed: E_u(L) = 0 reproduces the cubic Klein-Gordon equation!")

## 2. Lie Point Symmetry Generators

Using `infinitesimals(ansatz_degree=1)` computes the exact 4-dimensional Lie algebra:

In [ ]:
sol = infinitesimals(ckg_eq, u, (x, t), ansatz_degree=1)
print(f"Dimension of the symmetry algebra: {sol.dimension}\n")

labels = [
    "X_1 (Space Translation):",
    "X_2 (Time Translation):",
    "X_3 (Lorentz Boost):",
    "X_4 (Conformal Scale Invariance):",
]

for label, gen in zip(labels, sol.basis):
    is_valid = verify_generator(ckg_eq, u, (x, t), gen)
    print(f"{label}")
    print(f"  xi^x = {gen.xi[0]},  xi^t = {gen.xi[1]},  phi^u = {gen.phi[0]}")
    print(f"  Verified invariant: {is_valid}\n")

## 3. Singular Algebraic Traveling Profile

Under traveling wave reduction $\xi = \frac{x - v t}{\sqrt{1 - v^2}}$ with speed $|v| < 1$, the PDE admits the algebraic profile:
$$u(x, t) = -\frac{\sqrt{2}}{\xi} = -\frac{\sqrt{2(1 - v^2)}}{x - v t}$$

This is not a kink or a regular localized solitary wave: it is singular on $x=vt$ and is defined only away from that line.

In [ ]:
v_vel = sp.symbols("v", positive=True)
singular_profile = -sp.sqrt(2 * (1 - v_vel**2)) / (
    x - v_vel * t
)  # algebraic traveling-wave solution

# Exact scaling solution verification
residual = sp.simplify(ckg_eq.subs(u, singular_profile).doit())
print("Residual of singular traveling profile:", residual)
assert residual == 0
print("Verification: The singular profile satisfies the equation away from x = v*t.")